# 10 — Pilot-based GPU time and resource planner

**Purpose:** Estimate quota needs from REAL pilot timings; report assumptions instead of pretending estimates were observed full-run durations.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# Kaggle/Colab project setup — NO PIP INSTALLATIONS.
# Uses the versions already installed in this runtime.
from pathlib import Path
from importlib import metadata
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PULL_LATEST = False  # Enable only for a clean clone after pushing fixes to GitHub.
if Path("/kaggle/working").is_dir():
    WORKSPACE = Path("/kaggle/working")
elif Path("/content").is_dir():
    WORKSPACE = Path("/content")
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / "nlp_research_proj"
ROOT = REPO_DIR / "Urdu_Qwen_Research_Notebooks"

if (REPO_DIR / ".git").is_dir():
    if PULL_LATEST:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git folder: {REPO_DIR}; inspect before removing it")
elif (Path.cwd() / "project_utils.py").is_file():
    ROOT = Path.cwd()
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(f"Project code missing from {ROOT}")

# Check user-confirmed Kaggle versions; NEVER uninstall or upgrade preinstalled dependencies.
EXPECTED = {
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "torch": "2.11.0",  # the Kaggle build has a +cu128 suffix
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
}
versions = {}
for name, expected in EXPECTED.items():
    try:
        got = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Missing preinstalled dependency: {name}") from None
    versions[name] = got
    if got.split("+")[0] != expected:
        raise RuntimeError(
            f"This code is checked for {name}={expected}, found {got}. "
            "Do NOT run pip; use the tested Kaggle image or review the API changes."
        )
for name in ("accelerate", "datasets", "pandas", "matplotlib", "pytest"):
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Required package missing: {name}. This runtime cannot run the selected notebook.") from None
print("Using preinstalled Kaggle libraries (no installations):", versions)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
api = check_training_api()
print("Project ready:", ROOT)
print("Training API ready:", api["transformers"], api["trainer_tokenizer_key"], api["warmup_key"])


Full fine-tuning cannot be assigned one universal duration. Batch size, GPU model, token length, optimizer, and startup overhead matter. **No timing value below is a published benchmark until your own pilot runs.**

In [ ]:
import math, pandas as pd
pilots=[r for r in read_jsonl(ROOT/'outputs/metrics/train_runs.jsonl') if r.get('pilot')]
if not pilots:
    print('No pilot measurements yet. Run GPU pilot in notebook 00 first.')
else:
    out=[]
    for r in pilots:
        for examples_per_class in [10,25,50]:
            n=7*examples_per_class
            effective_batch=cfg()['per_device_batch_size']*cfg()['gradient_accumulation']
            estimated_steps=math.ceil(n/effective_batch)*cfg()['train_epochs']
            # Estimate from completed actual training timings. Warmup bias is possible.
            sps=r['wall_seconds']/max(r['global_steps'],1)
            out.append(dict(pilot=r['run_id'], method=r['method'],per_class=examples_per_class,
                            approx_optimizer_steps=estimated_steps,naive_minutes=round(sps*estimated_steps/60,1),
                            warmup_included=True,estimated_from_pilot_steps=r['global_steps']))
    display(pd.DataFrame(out))
    print('Use 10–20 steps for a steadier speed measurement; 2-step pilots include heavy warmup.')

### Practical quota and checkpoint advice

- For 2–3-epoch runs, start at 10/class and move to 25/class only after both methods work.
- Save checkpoints after **each finished run** to persistent storage (Google Drive on Colab, Kaggle dataset/output).
- A free notebook session can disconnect; saved files in ephemeral workspace may disappear.
- If you need to preserve checkpoints through interruptions, implement step-based checkpoints and resume in a dedicated long-run notebook. This project currently defaults to saving only completed runs to reduce storage.
- A 2-step pilot alone does not measure model download time or full training duration reliably.
- Measure `nvidia-smi` and peak `torch.cuda.max_memory_allocated()`; the latter is allocated memory, not full driver-level device utilization.